### **Project Introduction**
**Project Title:** AI Math Tutor

**Objective:**
The AI Math Tutor is a Python-based application designed to help students solve mathematical problems. It accepts user input in text format (like equations or expressions) and provides:

1. The correct solution.
2. A step-by-step explanation of the method used.

**Technologies Used:**
* SymPy: For precise symbolic mathematics (solving equations, derivatives, integrals).
* Gradio: To create an interactive web-based user interface.
* Google Colab: As the development and deployment environment.

In [ ]:
# Step 2: Install Required Libraries
# We need Gradio for the interface.
# SymPy is usually pre-installed in Colab, but we ensure it is available.

!pip install -q gradio sympy

**Import Libraries**

We need to import the necessary Python modules to handle the logic and the interface.

* sympy: The core math engine.
* gradio: For the UI.
* parse_expr: A special tool to convert text strings (like "2x") into math code.

In [ ]:
# Step 3: Import Libraries
import sympy as sp
import gradio as gr
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application

# Define the variable 'x' as the standard variable for our equations
x = sp.symbols('x')

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


**Input Preprocessing**

Users often type math problems in a way that is natural to write but incorrect in code. For example, typing 2x instead of 2*x or ^ instead of **.

This step creates a function to "clean" the input so the computer understands it correctly.

In [ ]:
# Step 4: User Input Preprocessing Function

def preprocess_input(user_input):
    """
    Converts a text string into a SymPy mathematical expression.
    """
    try:
        # 1. Replace caret (^) with double asterisks (**) for exponents
        user_input = user_input.replace('^', '**')

        # 2. Define transformations: This allows '2x' to be read as '2*x'
        transformations = standard_transformations + (implicit_multiplication_application,)

        # 3. Parse the string into a math expression
        expr = parse_expr(user_input, transformations=transformations)

        return expr

    except Exception as e:
        # Return an error message if the input is unreadable
        return f"Input Error: Could not understand the format. ({e})"

**AI Math Solver Logic**

This is the "Brain" of the tutor. We will create a function that takes the math problem and the category (Algebra, Calculus, etc.) and decides which mathematical operation to perform.


In [ ]:
# Step 5: Core Math Solver Logic

def solve_math_problem(problem_type, user_input):
    """
    Takes the problem type and user input, then returns a formatted solution.
    """
    # Check if input is empty
    if not user_input.strip():
        return "Please enter a math problem."

    # Preprocess the input
    expr = preprocess_input(user_input)

    # Check if preprocessing returned an error string
    if isinstance(expr, str) and "Error" in expr:
        return expr

    # Start building the output explanation
    output = f"### 📝 Problem Received:\n`{user_input}`\n\n---\n\n"

    try:
        # --- ALGEBRA LOGIC ---
        if problem_type == "Algebra (Solve Equation)":
            output += "**Step 1: Interpret Equation**\n"
            output += f"We are solving for x in: `{expr} = 0`\n\n"

            output += "**Step 2: Apply Algebraic Methods**\n"
            solutions = sp.solve(expr, x)

            output += f"**✅ Final Answer:**\n`x = {solutions}`"

        # --- DERIVATIVE LOGIC ---
        elif problem_type == "Derivative (Calculus)":
            output += "**Step 1: Identify Function**\n"
            output += f"`f(x) = {expr}`\n\n"

            output += "**Step 2: Apply Differentiation Rules**\n"
            derivative = sp.diff(expr, x)

            output += f"**✅ Final Answer:**\n`f'(x) = {derivative}`"

        # --- INTEGRATION LOGIC ---
        elif problem_type == "Integration (Calculus)":
            output += "**Step 1: Identify Function**\n"
            output += f"`f(x) = {expr}`\n\n"

            output += "**Step 2: Apply Integration Rules**\n"
            integral = sp.integrate(expr, x)

            output += f"**✅ Final Answer:**\n`∫f(x)dx = {integral} + C`"

        # --- SIMPLIFICATION LOGIC ---
        elif problem_type == "Simplification":
            output += "**Step 1: Analyze Expression**\n"
            simplified = sp.simplify(expr)

            output += f"**✅ Final Answer:**\n`{simplified}`"

        else:
            output = "Please select a valid problem type from the dropdown."

        return output

    except Exception as e:
        return f"⚠️ **Solver Error:** I couldn't solve this. Please check your input.\nDetails: `{e}`"

**Building the User Interface**

We will use **Gradio** to create a simple web app inside the notebook.
The interface will have:

1. A **Dropdown** to select the type of math problem.
2. A **Textbox** for the user to type their equation.
3. A **Button** to submit the problem.
4. An **Output** Area to display the step-by-step solution.


In [ ]:
# Step 6: Build and Launch Gradio Interface

# Define the theme and layout
with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue")) as tutor_interface:

    # Title
    gr.Markdown("# 🤖 AI Math Tutor")
    gr.Markdown("Welcome! Enter your math problem below. Use 'x' as your variable (e.g., x**2 + 2x).")

    # Row for Inputs
    with gr.Row():
        # Left Column: Inputs
        with gr.Column(scale=1):
            math_type = gr.Dropdown(
                choices=["Algebra (Solve Equation)", "Derivative (Calculus)", "Integration (Calculus)", "Simplification"],
                label="Select Problem Category",
                info="Choose the type of math operation"
            )
            user_input = gr.Textbox(
                label="Your Math Problem",
                placeholder="Example: x**2 - 4, or sin(x)"
            )
            solve_btn = gr.Button("Solve It 🚀", variant="primary")

        # Right Column: Output
        with gr.Column(scale=2):
            result_display = gr.Markdown(label="Solution")

    # Connect the button to the solver function
    solve_btn.click(
        fn=solve_math_problem,
        inputs=[math_type, user_input],
        outputs=result_display
    )

    # Add Example Buttons for Beginners
    gr.Markdown("### 🧪 Try These Examples:")
    gr.Examples(
        examples=[
            ["Algebra (Solve Equation)", "x**2 - 9"],
            ["Derivative (Calculus)", "3*x**4 - 2*x"],
            ["Integration (Calculus)", "2*x + 1"],
            ["Simplification", "(x**2 - 1) / (x - 1)"]
        ],
        inputs=[math_type, user_input]
    )

print("🚀 Launching Interface... (Click the public link below)")

/tmp/ipykernel_268/3180233203.py:4: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue")) as tutor_interface:


🚀 Launching Interface... (Click the public link below)


**Launch the Application**

Run the cell below to start the AI Math Tutor.
**Note:** Google Colab will provide a public link (ending in .gradio.live) after a few seconds. Click that link to open the tutor in a new tab.

In [ ]:
# Step 7: Launch the App
# share=True creates a public link that works anywhere
# debug=True helps us see errors in the console if they happen

tutor_interface.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3065741be650cd306c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Testing and Results**

You should now see a working interface.

**Example Test Cases:**
1. Algebra: Type x^2 - 4 (it will auto-convert ^ to power). Select "Algebra". Result should be x = [-2, 2].
2. Derivative: Type x^3. Select "Derivative". Result should be 3x^2.
3. Simplification: Type (x^2 - 1) / (x - 1). Select "Simplification". Result should be x + 1.

**Future Improvements**

1. **Graphing:** Add a plotter to visualize the equation solution.

2. **Natural Language:** Allow users to type "solve for x" instead of just the equation.
3. **Handwriting Recognition:** Allow users to upload images of handwritten math problems.


